# LeetCode 35 — Search Insert Position | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Sorted Array · Binary Search · lower_bound  
**Bài gốc:** https://leetcode.com/problems/search-insert-position/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
class Solution:
    def searchInsert(self, nums: list[int], target: int) -> int:
        ...
```

**Sau bài học, bạn có thể:**
- Hiểu đúng vị trí chèn khi target chưa tồn tại.
- Tự suy luận từ quét tuyến tính sang tìm kiếm nhị phân.
- Giải thích bất biến khoảng [left,right), độ phức tạp O(log n).

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Đề bài và những ví dụ phải phân biệt

Cho mảng số nguyên `nums` đã **sắp xếp tăng** và **không trùng**, cùng `target`.

- Nếu target **đã có**, trả chỉ số của nó.
- Nếu **chưa có**, trả chỉ số cần **chèn target** để mảng vẫn tăng.

| nums | target | đáp án | giải thích |
|---|---:|---:|---|
| `[1,3,5,6]` | 5 | 2 | chính là số ở index 2 |
| `[1,3,5,6]` | 2 | 1 | chen giữa 1 và 3 |
| `[1,3,5,6]` | 7 | 4 | sau phần tử cuối |
| `[1,3,5,6]` | 0 | 0 | trước phần tử đầu |

Cốt lõi: tìm **vị trí đầu tiên có số >= target**; nếu không có thì trả `len(nums)`.

In [ ]:
nums=[1,3,5,6]
for target in [5,2,7,0]:
    print("target",target,"-> vị trí chèn trực giác",sum(x<target for x in nums))

## 2. Nền tảng Python: list, index, các toán tử

`nums[i]` lấy giá trị ở index `i` (bắt đầu từ 0). `len(nums)` là số phần tử. `>=` nghĩa là lớn hơn **hoặc** bằng; `>` không tính trường hợp bằng, nên dễ tạo lỗi vị trí khi target tồn tại. `//` là **chia lấy phần nguyên** để tính vị trí giữa.

In [ ]:
nums=[1,3,5,6]
print("chỉ số:",list(range(len(nums))))
print("nums[2] =",nums[2]);print("5 >= 5?",5>=5)
print("(1+4)//2 =",(1+4)//2)

## 3. Cách dễ nghĩ: đi từ trái qua phải

Duyệt từng phần tử; gặp **giá trị đầu tiên >= target** thì trả index. Nếu duyệt hết chưa thấy, target lớn hơn mọi số → trả `len(nums)`.

Lời giải này đúng và rất dễ nhớ, nhưng **O(n)**: với mảng một triệu phần tử và target ở cuối, ta vẫn phải nhìn gần như cả triệu số.

In [ ]:
def search_insert_linear(nums:list[int],target:int)->int:
    for i,x in enumerate(nums):
        if x>=target:
            return i
    return len(nums)

for t in [0,2,5,7]: print(t,"=>",search_insert_linear([1,3,5,6],t))

## 4. Insight: vì mảng đã sorted, bỏ được một nửa

Ở vị trí `mid`, nếu `nums[mid] < target` thì **toàn bộ từ đầu đến mid** nhỏ hơn target: chúng không thể là đáp án. Ta bỏ cả nửa trái bằng `left=mid+1`.

Ngược lại, nếu `nums[mid]>=target`, **mid có thể là đáp án**, nên không được xóa mid: giữ lại bằng `right=mid`.

Đây là **Binary Search** (tìm kiếm nhị phân): mỗi lượt cắt khoảng cần xét còn khoảng một nửa.

In [ ]:
nums=[1,3,5,6,9,11,15]
left,right=0,len(nums);mid=(left+right)//2
print("[left,right) =",(left,right),"; mid=",mid,"; nums[mid]=",nums[mid])
print("Nếu target=4, nums[mid] >=4 -> right=mid (giữ mid)")

## 5. Tại sao viết khoảng `[left, right)`?

Ký hiệu `[left,right)` là **gồm left, không gồm right**. Ban đầu `left=0`, `right=len(nums)` để bao phủ toàn bộ các index 0..n-1.

**Bất biến quan trọng:**

1. Mọi phần tử ở **trước left** đều `< target`.
2. Mọi phần tử từ **right trở đi** đều `>= target`.
3. Vị trí cần tìm nằm giữa hai biên `left` và `right`, kể cả trường hợp vị trí chèn là `n`.

Khi `left==right`, đó chính là vị trí đầu tiên có giá trị >= target.

In [ ]:
nums=[1,3,5,6]
left,right=0,len(nums)
print("Vùng chưa xét:",nums[left:right],"với [",left,",",right,")")
left,right=1,2
print("Thu hẹp:",nums[left:right],"chỉ xét index 1")

## 6. Công thức vòng lặp — từng nhánh một

Khi `left < right`, còn ít nhất một vị trí chưa xét:

- `mid=(left+right)//2` là giữa khoảng.
- Nếu `nums[mid] < target`: loại **mid và trái**, đặt `left=mid+1`.
- Ngược lại: giữ `mid`, loại phần **sau** mid, đặt `right=mid`.

**Tại sao không đặt `left=mid`?** Với khoảng chỉ có 1 phần tử, `mid==left`; nếu tiếp tục dùng `left=mid` thì vòng lặp có thể lặp vô hạn.

In [ ]:
left,right=1,2
mid=(left+right)//2
print("left,right,mid=",left,right,mid)
print("Sai nếu left=mid vì left vẫn là",mid)
print("Đúng nếu nums[mid]<target -> left=mid+1 =",mid+1)

## 7. Thuật toán chính — tìm lower bound

**Lower bound** nghĩa là vị trí đầu tiên mà phần tử không nhỏ hơn target. Đây chính là chỉ số chèn. Hàm không cần thực sự chèn vào mảng; chỉ trả số nguyên.

In [ ]:
def search_insert_binary(nums:list[int],target:int)->int:
    left,right=0,len(nums)
    while left<right:
        mid=(left+right)//2
        if nums[mid]<target:
            left=mid+1
        else:
            right=mid
    return left

for target in [5,2,7,0]:
    print(target,"->",search_insert_binary([1,3,5,6],target))

## 8. Bảng trace: `[1,3,5,6]`, target=2

| Lượt | left | right | mid | nums[mid] | hành động |
|---:|---:|---:|---:|---:|---|
| 1 | 0 | 4 | 2 | 5 | 5>=2 → `right=2` |
| 2 | 0 | 2 | 1 | 3 | 3>=2 → `right=1` |
| 3 | 0 | 1 | 0 | 1 | 1<2 → `left=1` |
| dừng | 1 | 1 | – | – | `return 1` |

**Nhìn kỹ:** khi `nums[mid]>=target`, không được cộng/trừ 1 làm mất ứng viên `mid`.

In [ ]:
def trace_search_insert(nums,target):
    left,right=0,len(nums); step=0
    while left<right:
        step+=1;mid=(left+right)//2
        print(f"Lượt {step}: left={left},right={right},mid={mid},nums[mid]={nums[mid]}")
        if nums[mid]<target:
            left=mid+1; print("  nhỏ hơn -> left=",left)
        else:
            right=mid; print("  đủ lớn -> right=",right)
    print("Dừng, đáp án:",left);return left

assert trace_search_insert([1,3,5,6],2)==1

## 9. Vì sao thuật toán không bỏ sót đáp án?

Nếu `nums[mid]<target`, mọi phần tử trước và tại mid đều quá nhỏ, bỏ đi là an toàn. Nếu `nums[mid]>=target`, mid hoặc một phần tử bên trái có thể là vị trí đầu tiên nên phải giữ mid. Mỗi lần vòng lặp thu nhỏ khoảng, vẫn giữ đáp án nằm trong `[left,right]`. Khi khoảng rỗng (`left==right`), vị trí duy nhất còn lại là đáp án.

## 10. Time / Space — so với quét tuyến tính

`n = len(nums)`. Quét tuyến tính có **Time O(n)**, **Space O(1)**. Binary search giảm một nửa số vị trí mỗi lượt: sau k lượt còn khoảng n/2^k → khi còn 1, k xấp xỉ log₂(n). Vì thế **Time O(log n)**, **Space O(1)**. Đây là lợi thế quan trọng của mảng **đã sắp xếp**.

In [ ]:
for n in [8,1024,1_048_576]:
    left,right=0,n;count=0
    while left<right:
        mid=(left+right)//2
        left=mid+1;count+=1
    print("n=",n,"số lượt tối đa kiểu này≈",count)

## 11. Code nộp LeetCode chuẩn

Chú ý signature **`searchInsert`** (chữ I viết hoa), trả **int**. Không thay đổi `nums`; đề chỉ hỏi vị trí.

In [ ]:
class Solution:
    def searchInsert(self, nums:list[int], target:int)->int:
        left,right=0,len(nums)
        while left<right:
            mid=(left+right)//2
            if nums[mid]<target:
                left=mid+1
            else:
                right=mid
        return left

assert Solution().searchInsert([1,3,5,6],5)==2

## 12. Kiểm thử ví dụ, biên và phản ví dụ

Thử target nhỏ nhất/lớn nhất, đúng bằng phần tử, chen ở giữa, mảng một phần tử. Bài gốc nói `nums` không rỗng; code này cũng tự xử lý danh sách rỗng, đó là **mở rộng**, không phải constraints bắt buộc.

In [ ]:
cases=[([1,3,5,6],5,2),([1,3,5,6],2,1),([1,3,5,6],7,4),
       ([1,3,5,6],0,0),([1],1,0),([1],0,0),([1],2,1),
       ([1,2,4,7],3,2),([],5,0)]
for nums,target,want in cases:
    assert Solution().searchInsert(nums,target)==want,(nums,target)
    assert search_insert_linear(nums,target)==want
print("PASS",len(cases),"trường hợp")

## 13. Dùng `bisect_left` làm đối chiếu

Python có thư viện chuẩn `bisect`. Hàm `bisect_left(nums,target)` trả chính xác vị trí **đầu tiên** có giá trị >= target. Ở môi trường làm việc nó rất hữu ích; trong buổi học này ta tự viết binary search để hiểu bản chất.

In [ ]:
from bisect import bisect_left
for t in [0,2,5,7]:
    got=bisect_left([1,3,5,6],t)
    assert got==search_insert_binary([1,3,5,6],t)
    print(t,"->",got)

## 14. Sai lầm kinh điển

❌ Dùng `while left<=right` với khoảng `[left,right)` rồi đọc `nums[mid]` khi `mid==len(nums)` có thể **IndexError**.

❌ Cập nhật `right=mid-1` làm mất một mid đang là ứng viên; đó là **mẫu tìm kiếm khác**, cần invariant khác.

❌ Dùng `if nums[mid] <= target` sẽ tìm vị trí **sau** target thay vì vị trí target.

✅ Luôn nhớ: **`[left,right)`, `while left<right`, `< -> left=mid+1`, còn lại -> `right=mid`**.

In [ ]:
nums=[1,3,5,6]; target=5
print("search_insert đúng:",search_insert_binary(nums,target))
print("mảng nếu chèn vào vị trí đó:",nums[:2]+[target]+nums[2:])

## 15. Mẫu thuật toán có thể dùng lại

Binary search phù hợp khi một điều kiện thay đổi **một lần** từ False → True theo chỉ số. Trong bài này `nums[i] >= target` đúng từ vị trí chèn trở về sau. Tìm **True đầu tiên** = lower bound. Đây là kỹ năng sẽ gặp lại ở bài 34, 69 và nhiều bài Medium.

## 16. Năm bài tự luyện — đừng chạy các hàm TODO khi chưa làm

### Bài 1 — tính mid — Tự làm

Viết hàm tính `mid` trong đoạn `[left,right)`.

**Gợi ý cấp 1 (tư duy):** Lấy chỉ số giữa.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng phép chia nguyên `//`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`(left+right)//2`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_35_1(left:int,right:int)->int:
    # TODO
    pass

# assert exercise_35_1(0,5)==2

### Bài 2 — tính bằng tay — Tự làm

`nums=[1,3,5,6]`, target=4, vị trí cần chèn là mấy?

**Gợi ý cấp 1 (tư duy):** Nằm sau 3 và trước 5.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Chỉ số của số 5 là 2.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Trả 2.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_35_2()->int:
    # TODO
    pass

# assert exercise_35_2()==2

### Bài 3 — quét tuyến tính — Tự làm

Tự viết bản O(n), tìm vị trí đầu tiên >=target.

**Gợi ý cấp 1 (tư duy):** Duyệt từ trái.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng enumerate.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Không tìm thấy thì return len(nums).

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_35_3(nums:list[int],target:int)->int:
    # TODO
    pass

# assert exercise_35_3([1,3,5],4)==2

### Bài 4 — binary search — Tự làm

Tự viết lại thuật toán nửa mở đã học.

**Gợi ý cấp 1 (tư duy):** Giữ invariant về left/right.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

while left<right; mid=(left+right)//2.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Nếu nums[mid]<target: left=mid+1; ngược lại right=mid.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_35_4(nums:list[int],target:int)->int:
    # TODO
    pass

# assert exercise_35_4([1,3,5,6],7)==4

### Bài 5 — đếm số nhỏ hơn target — Tự làm

Trả số phần tử `< target` trong mảng sorted. **Được phép có phần tử trùng** (biến thể ngoài bài gốc).

**Gợi ý cấp 1 (tư duy):** Số lượng nhỏ hơn chính là index đầu tiên đủ lớn.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng lower bound.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Trả chỉ số tìm bằng code bài 35.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_35_5(nums:list[int],target:int)->int:
    # TODO
    pass

# assert exercise_35_5([1,2,2,5],3)==3

## 17. Đáp án tham khảo — chỉ xem sau khi thử

In [ ]:
def answer_35_1(l,r):return (l+r)//2
def answer_35_2():return 2
def answer_35_3(nums,target):
    for i,x in enumerate(nums):
        if x>=target:return i
    return len(nums)
def answer_35_4(nums,target):
    l,r=0,len(nums)
    while l<r:
        m=(l+r)//2
        if nums[m]<target:l=m+1
        else:r=m
    return l
def answer_35_5(nums,target):return answer_35_4(nums,target)
assert answer_35_1(0,5)==2 and answer_35_2()==2
assert answer_35_3([1,3,5],4)==2
assert answer_35_4([1,3,5,6],7)==4
assert answer_35_5([1,2,2,5],3)==3
print("Đáp án PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu nhớ:** `left=0; right=n; while left<right: mid=(left+right)//2; if nums[mid]<target: left=mid+1; else: right=mid; return left`. Thời gian **O(log n)**, bộ nhớ **O(1)**.

## Checklist trước khi submit

- [ ] Tôi giải thích được tại sao nửa mở right=n.
- [ ] Tôi không nhầm khi target vắng mặt.
- [ ] Tôi trace được từng mid của ví dụ target=2.
- [ ] Tôi dùng `<` (không dùng `<=`) để tìm lower bound.
- [ ] Tôi đã test target đầu, cuối, giữa, không tồn tại.

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*